In [1]:
"""
DES (Data Encryption Standard) 블록암호 실습

- 64비트(8바이트) 블록 단위로 암·복호화
- 유효 키 길이: 56비트 (8바이트 키 중 패리티 8비트 포함)
- 16라운드 Feistel 구조
- 교육용: 실제 보안 목적에는 3DES·AES 사용 권장

필요 패키지:
  python -m pip install pycryptodome
"""

from __future__ import annotations

try:
    from Crypto.Cipher import DES
    from Crypto.Util.Padding import pad, unpad
except ImportError as e:  # pragma: no cover
    raise ImportError(
        "pycryptodome이 필요합니다. 터미널에서: python -m pip install pycryptodome"
    ) from e


def normalize_des_key(key: str | bytes) -> bytes:
    """DES 키를 정확히 8바이트로 맞춤 (부족 시 0 패딩, 초과 시 잘라냄)."""
    if isinstance(key, str):
        key = key.encode("utf-8")
    if len(key) < 8:
        key = key + b"\x00" * (8 - len(key))
    elif len(key) > 8:
        key = key[:8]
    return key


def encrypt_ecb(plaintext: str, key: str | bytes) -> bytes:
    """ECB 모드 암호화 (교육용 데모)."""
    k = normalize_des_key(key)
    cipher = DES.new(k, DES.MODE_ECB)
    data = pad(plaintext.encode("utf-8"), DES.block_size)
    return cipher.encrypt(data)


def decrypt_ecb(ciphertext: bytes, key: str | bytes) -> str:
    """ECB 모드 복호화."""
    k = normalize_des_key(key)
    cipher = DES.new(k, DES.MODE_ECB)
    data = unpad(cipher.decrypt(ciphertext), DES.block_size)
    return data.decode("utf-8")


def encrypt_cbc(plaintext: str, key: str | bytes, iv: bytes) -> bytes:
    """CBC 모드 암호화 (IV는 DES 블록 크기 8바이트)."""
    if len(iv) != 8:
        raise ValueError("IV는 DES 블록 크기와 같아야 합니다 (8바이트).")
    k = normalize_des_key(key)
    cipher = DES.new(k, DES.MODE_CBC, iv=iv)
    data = pad(plaintext.encode("utf-8"), DES.block_size)
    return cipher.encrypt(data)


def decrypt_cbc(ciphertext: bytes, key: str | bytes, iv: bytes) -> str:
    """CBC 모드 복호화."""
    if len(iv) != 8:
        raise ValueError("IV는 8바이트여야 합니다.")
    k = normalize_des_key(key)
    cipher = DES.new(k, DES.MODE_CBC, iv=iv)
    data = unpad(cipher.decrypt(ciphertext), DES.block_size)
    return data.decode("utf-8")


def demo() -> None:
    secret = "my8key!"  # 8글자 -> 8바이트(UTF-8) 키
    message = "Hello DES - symmetric block cipher."

    print("=== DES 실습 (PyCryptodome) ===\n")

    # ECB
    ct_ecb = encrypt_ecb(message, secret)
    print("[ECB] 암호문 (hex):", ct_ecb.hex())
    print("[ECB] 복호문:", decrypt_ecb(ct_ecb, secret))

    # CBC
    iv = b"\x01" * 8  # 데모용 고정 IV (실무는 os.urandom(8) 권장)
    ct_cbc = encrypt_cbc(message, secret, iv)
    print("\n[CBC] IV (hex):", iv.hex())
    print("[CBC] 암호문 (hex):", ct_cbc.hex())
    print("[CBC] 복호문:", decrypt_cbc(ct_cbc, secret, iv))

    # ECB 한계: 동일 평문 블록 반복 -> 동일 암호문 블록
    block = "ABCDEFGH"  # 정확히 8바이트(ASCII)
    pt_raw = (block * 2).encode("utf-8")
    # encrypt_ecb()는 내부에서 PKCS7 패딩을 적용하므로,
    # 여기서도 동일한 방식으로 패딩된 블록을 같이 관찰합니다.
    pt_padded = pad(pt_raw, DES.block_size)
    pt_blocks = [pt_padded[i : i + DES.block_size] for i in range(0, len(pt_padded), DES.block_size)]

    ct_ecb_full = encrypt_ecb(block * 2, secret)
    ct_blocks = [ct_ecb_full[i : i + DES.block_size] for i in range(0, len(ct_ecb_full), DES.block_size)]

    print("\n[ECB 한계] 동일 8바이트 평문 블록이 암호문에서 어떻게 반복되는지")
    print("Plaintext blocks (hex):")
    for idx, b in enumerate(pt_blocks):
        print(f"  P{idx}: {b.hex()}")
    print("Ciphertext blocks (hex):")
    for idx, b in enumerate(ct_blocks):
        print(f"  C{idx}: {b.hex()}")

    # 앞의 두 블록은 원래 평문이 동일(패딩 블록은 C2에 해당)
    print("\n동일 평문 블록(1,2) -> 동일 암호문 블록(0,1)?", ct_blocks[0] == ct_blocks[1])


if __name__ == "__main__":
    demo()



=== DES 실습 (PyCryptodome) ===

[ECB] 암호문 (hex): bd6b71f2a4b673a98369db01aeb12d9c32660037028e35842a55877634f7a2cea74c736e2f1eab99
[ECB] 복호문: Hello DES - symmetric block cipher.

[CBC] IV (hex): 0101010101010101
[CBC] 암호문 (hex): 40251532069aaa2781fa35af5a0b5479db9fbc865ded09c3479702847b5aa8f3cd51d966b1da6d7a
[CBC] 복호문: Hello DES - symmetric block cipher.

[ECB 한계] 동일 8바이트 평문 블록이 암호문에서 어떻게 반복되는지
Plaintext blocks (hex):
  P0: 4142434445464748
  P1: 4142434445464748
  P2: 0808080808080808
Ciphertext blocks (hex):
  C0: 17b7862c57fc6279
  C1: 17b7862c57fc6279
  C2: d9751a468e040dc0

동일 평문 블록(1,2) -> 동일 암호문 블록(0,1)? True
